# 02 · Os recursos da série 1.x  ⏱ 25 min

A série 1.x (jun/2024 a mar/2026) estabilizou o formato de arquivo e trouxe boa parte do que torna o DuckDB produtivo no dia a dia:

1. Ler e escrever arquivos de qualquer lugar
2. SQL amigável (*friendly SQL*)
3. Janelas, `QUALIFY`, `PIVOT`
4. `ASOF JOIN`
5. `MERGE INTO`, `TRY`, `FILL` (1.3 e 1.4)
6. Tipos aninhados e lambdas
7. Extensões e dados remotos

In [ ]:
import duckdb
con = duckdb.connect("curso.duckdb")

## 1. Ler e escrever arquivos

### CSV com detecção automática
O *sniffer* descobre separador (aqui é `;`), cabeçalho e tipos.

In [ ]:
con.sql("DESCRIBE FROM 'dados/vendas_2025_01.csv'").show()

In [ ]:
# Opções explícitas quando precisar
con.sql("""
    FROM read_csv('dados/vendas_2025_01.csv', delim=';', header=true,
                  types={'canal': 'VARCHAR'}, sample_size=-1)
    LIMIT 3
""").show()

### Vários arquivos com glob, e a coluna `filename`

In [ ]:
con.sql("""
    SELECT filename, count(*) AS linhas
    FROM read_json('dados/logs/*.jsonl', filename=true)
    GROUP BY ALL ORDER BY ALL
""").show()

### Partições Hive: `ano=2025/mes=1/…`
O filtro em `ano`/`mes` faz o DuckDB **pular pastas inteiras** (*partition pruning*).

In [ ]:
con.sql("""
    SELECT ano, mes, count(*) AS itens
    FROM read_parquet('dados/lake/vendas/*/*/*.parquet', hive_partitioning=true)
    WHERE ano = 2025 AND mes <= 3
    GROUP BY ALL ORDER BY ALL
""").show()

### Escrever: `COPY … TO`
Converter CSV em Parquet é uma linha.

In [ ]:
con.sql("COPY (FROM 'dados/vendas_2025_01.csv') TO 'dados/vendas_2025_01.parquet' (FORMAT parquet)")
import os
print(f"CSV:     {os.path.getsize('dados/vendas_2025_01.csv')/1e6:.1f} MB")
print(f"Parquet: {os.path.getsize('dados/vendas_2025_01.parquet')/1e6:.1f} MB")

## 2. SQL amigável

| Recurso | Exemplo |
|---|---|
| `FROM` primeiro | `FROM vendas LIMIT 5` |
| `GROUP BY ALL` / `ORDER BY ALL` | agrupa por tudo que não é agregado |
| `SELECT * EXCLUDE / REPLACE / RENAME` | tira, troca ou renomeia colunas |
| `COLUMNS(regex)` | aplica a mesma expressão a várias colunas |
| Alias como prefixo (1.2) | `SELECT receita: sum(x)` |
| Alias reaproveitado | usar o alias no `WHERE` e no `GROUP BY` |
| `SUMMARIZE` | perfil estatístico da tabela |

In [ ]:
con.sql("FROM vendas LIMIT 3").show()

In [ ]:
con.sql("""
    SELECT * EXCLUDE (venda_id, pedido_id)
             REPLACE (round(preco_unitario) AS preco_unitario)
    FROM vendas LIMIT 3
""").show()

In [ ]:
# alias como prefixo + alias usado em outras cláusulas + GROUP BY ALL
con.sql("""
    SELECT
        mes:     date_trunc('month', data_venda),
        receita: round(sum(quantidade * preco_unitario)),
        ticket:  round(receita / count(DISTINCT pedido_id), 2)
    FROM vendas
    WHERE status = 'concluido'
    GROUP BY ALL
    ORDER BY mes
    LIMIT 6
""").show()

In [ ]:
# COLUMNS: a mesma agregação para várias colunas
con.sql("SELECT max(COLUMNS('quantidade|preco_unitario')), avg(COLUMNS('quantidade|preco_unitario')) FROM vendas").show()

In [ ]:
con.sql("SUMMARIZE produtos").df()

## 3. Janelas, `QUALIFY` e `PIVOT`

`QUALIFY` filtra pelo resultado de uma função de janela, sem subconsulta. Aqui: o produto mais vendido de cada categoria.

In [ ]:
con.sql("""
    SELECT p.categoria, p.nome, sum(v.quantidade) AS unidades
    FROM vendas v JOIN produtos p USING (produto_id)
    GROUP BY p.categoria, p.nome
    QUALIFY row_number() OVER (PARTITION BY p.categoria ORDER BY unidades DESC) = 1
    ORDER BY unidades DESC
""").show()

In [ ]:
# PIVOT: receita por categoria (linhas) x trimestre (colunas)
con.sql("""
    PIVOT (
        SELECT p.categoria,
               year(data_venda) || '-T' || quarter(data_venda) AS trimestre,
               quantidade * preco_unitario AS valor
        FROM vendas v JOIN produtos p USING (produto_id)
        WHERE year(data_venda) = 2025
    )
    ON trimestre USING round(sum(valor) / 1e6, 1)
    ORDER BY categoria
""").show()

## 4. `ASOF JOIN`: o valor vigente naquele momento

Cada venda deve ser comparada com o **preço de tabela que valia na data da venda**. Um JOIN normal exigiria a data exata. O `ASOF` pega a linha mais recente com `valido_desde <= data_venda`.

In [ ]:
con.sql("""
    SELECT v.venda_id, v.produto_id, v.data_venda, v.preco_unitario,
           h.valido_desde, h.preco_tabela,
           round(100 * (v.preco_unitario / h.preco_tabela - 1), 1) AS desconto_pct
    FROM vendas v
    ASOF JOIN 'dados/precos_historico.parquet' h
        ON v.produto_id = h.produto_id AND v.data_venda >= h.valido_desde
    WHERE v.produto_id = 42
    ORDER BY v.data_venda
    LIMIT 5
""").show()

## 5. Novidades da 1.3 e 1.4: `TRY`, `MERGE INTO`, `FILL`

### `TRY(expr)` (1.3): erro vira `NULL`

In [ ]:
con.sql("SELECT TRY(CAST('abc' AS INTEGER)) AS a, TRY(CAST('123' AS INTEGER)) AS b").show()

### `MERGE INTO` (1.4): *upsert* sem precisar de chave primária
Atualiza o preço dos produtos que mudaram e insere os novos, numa só instrução.

In [ ]:
con.sql("CREATE OR REPLACE TABLE catalogo AS SELECT produto_id, nome, preco FROM produtos WHERE produto_id <= 5")
con.sql("""
    CREATE OR REPLACE TEMP TABLE atualizacao AS
    SELECT * FROM (VALUES
        (1, 'Produto 1 (novo nome)', 10.0),
        (2, NULL, 20.0),
        (999, 'Produto novo', 99.9)
    ) t(produto_id, nome, preco)
""")
con.sql("""
    MERGE INTO catalogo c
    USING atualizacao a USING (produto_id)
    WHEN MATCHED THEN UPDATE SET nome = coalesce(a.nome, c.nome), preco = a.preco
    WHEN NOT MATCHED THEN INSERT VALUES (a.produto_id, a.nome, a.preco)
    RETURNING merge_action, *
""").show()

### `FILL` (1.4): preencher lacunas em séries
Interpola valores nulos usando a ordem da janela.

In [ ]:
con.sql("""
    SELECT dia, temperatura, fill(temperatura) OVER (ORDER BY dia) AS preenchida
    FROM (VALUES (1, 20.0), (2, NULL), (3, NULL), (4, 26.0), (5, 25.0)) t(dia, temperatura)
""").show()

## 6. Tipos aninhados e lambdas

`LIST`, `STRUCT`, `MAP` são tipos de verdade, com acesso por ponto. Funções de lista aceitam lambdas.

> Na 2.0, a sintaxe antiga `x -> x + 1` vem **desligada por padrão**. Use `lambda x: x + 1`.

In [ ]:
con.sql("""
    SELECT nome, atributos.cor, atributos.peso_kg
    FROM produtos LIMIT 3
""").show()

In [ ]:
con.sql("""
    SELECT cliente_id,
           list(DISTINCT canal ORDER BY canal)                       AS canais,
           list_transform(list(quantidade), lambda q: q * 10)[1:3]   AS amostra_x10,
           list_filter(list(preco_unitario), lambda p: p > 1000).len() AS itens_caros
    FROM vendas
    WHERE cliente_id <= 3
    GROUP BY ALL ORDER BY ALL
""").show()

## 7. Extensões e dados remotos

Extensões são baixadas sob demanda (`INSTALL` uma vez, `LOAD` em cada sessão; muitas carregam sozinhas). As principais: `httpfs` (S3, HTTP, Azure, GCS), `spatial`, `postgres`, `mysql`, `sqlite`, `excel`, `iceberg`, `delta`, `ducklake`, `ui`.

> As células abaixo **precisam de internet**.

In [ ]:
try:
    con.sql("INSTALL httpfs; LOAD httpfs;")
    con.sql("""
        SELECT count(*) AS corridas, round(avg(total_amount), 2) AS ticket_medio
        FROM 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
    """).show()
except Exception as e:
    print("Sem internet ou extensão indisponível:", str(e).splitlines()[0])

Só as colunas e *row groups* necessários são baixados (HTTP range requests). Para S3 privado:

```sql
CREATE SECRET s3_empresa (TYPE s3, PROVIDER credential_chain);
SELECT * FROM 's3://meu-bucket/vendas/*.parquet';
```

### Banco criptografado (1.4)
```sql
ATTACH 'secreto.duckdb' AS sec (ENCRYPTION_KEY 'minha-chave');  -- AES-256-GCM
```
(Para gravar, a extensão `httpfs` precisa estar carregada, pois ela fornece o OpenSSL.)

In [ ]:
try:
    con.sql("ATTACH 'secreto.duckdb' AS sec (ENCRYPTION_KEY 'quack-quack')")
    con.sql("CREATE OR REPLACE TABLE sec.clientes_vip AS FROM clientes WHERE segmento = 'empresa' LIMIT 100")
    con.sql("SELECT count(*) FROM sec.clientes_vip").show()
    con.sql("DETACH sec")
except Exception as e:
    print("Precisa da httpfs carregada:", str(e).splitlines()[0][:120])

In [ ]:
con.close()